In [22]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print("libraries ready")

libraries ready


In [25]:
from groq import Groq
API_KEY = "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"
client = Groq(api_key = API_KEY)

MODEL = "llama-3.1-8b-instant"
print(f'Groq client configured with model {MODEL}')
print('Make sure API_KEY is replaced with your actual key')

Groq client configured with model llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key
Groq client configured with model llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key


In [26]:
def ask_llm(user_message, system_message = "You are a helpful assistant.", temperature = 0.7, max_tokens = 500):
  response = client.chat.completions.create(
      model = MODEL,
      messages = [
          {"role": "system", "content": system_message},
           {"role": "user", "content": user_message},
      ],
      temperature = temperature,
      max_tokens = max_tokens,
      stream = False,
  )
  return response.choices[0].message.content

test_response  = ask_llm("What is the capital of France?")

test_response2  = ask_llm("Is Gen AI and Data Engineering a good carrer in 2026")
print('===== LLM RESPONSE ====')
print(test_response)
print(test_response2)

===== LLM RESPONSE ====
The capital of France is Paris.
Based on current trends and market demand, Gen AI and Data Engineering are excellent career choices in 2026. Here's why:

**Gen AI:**

1. **Rapid growth:** The global AI market is expected to grow to $190 billion by 2025, with Gen AI being a key driver of this growth.
2. **High demand:** As AI adoption increases across industries, the demand for Gen AI experts is expected to rise, with a projected shortage of 20% by 2026.
3. **Low unemployment:** Gen AI professionals are in high demand, with a low unemployment rate compared to other tech fields.
4. **Competitive salaries:** Gen AI professionals can command high salaries, with median salaries ranging from $141,000 to over $250,000 depending on experience and location.

**Data Engineering:**

1. **Data explosion:** The amount of data generated worldwide is expected to reach 181 zettabytes by 2026, creating a massive demand for data engineers.
2. **Industry growth:** The global data 

In [27]:


# Ask about concepts from Phase 1 – showing LLM knows our domain
response_etl = ask_llm(
    "In 3 bullet points, explain how the Medallion Architecture "
    "(Bronze, Silver, Gold layers) relates to ETL pipelines.",
    system_message="You are a senior data engineering instructor. "
                   "Be concise and practical."
)
print('Medallion + ETL connection:')
print(response_etl)
print()
print('--- Token explanation ---')
print('Each word is roughly 1-2 tokens.')
print(f'The model above used approximately', len(response_etl.split())*1.3, 'tokens.')
print('Llama-3.1-8b context window: 8192 tokens (~6000 words per conversation)')


Medallion + ETL connection:
Here are 3 bullet points explaining how the Medallion Architecture relates to ETL (Extract, Transform, Load) pipelines:

• **Bronze Layer**: This is the landing zone for raw, unprocessed data. In ETL pipelines, the Bronze layer acts as the source system where data is first extracted from, often from various sources such as databases, APIs, or files. It's a highly raw, unfiltered, and untransformed version of the data.

• **Silver Layer**: This layer involves data processing, cleaning, and transformation. In ETL pipelines, the Silver layer is where you perform data quality checks, data validation, data normalization, and data aggregation. The transformed data is then loaded into this layer, making it a semi-processed version of the data.

• **Gold Layer**: This is the final, refined version of the data. In ETL pipelines, the Gold layer represents the processed data that has been completely transformed, filtered, and formatted for consumption by business users

In [28]:
zero_shot_response=ask_llm(
    "Extract the city name from this address:"
    "456 brigade Road, Banglore 560025, karnataka,India"
)
print("Zero-shot result: ")
print(zero_shot_response)
print()

ambigious_response=ask_llm("Clean this data :ramesh Kumar.45000.mumbai")

Zero-shot result: 
The city name is Bangalore.



In [29]:
zero_shot_response=ask_llm(
    "Extract the city name from this address:"
    "456 brigade Road, Banglore 560025, karnataka,India"
)
print("Zero-shot result: ")
print(zero_shot_response)

print()

ambigious_response=ask_llm("Clean this data :ramesh Kumar.45000.mumbai")
print('AmbigiousZero_shoy result:')
print(ambigious_response)
print()
print("the problem is unpredictable and not machine-parseable!")

Zero-shot result: 
The city name is "Banglore."

AmbigiousZero_shoy result:
The cleaned data would be:

Ramesh Kumar, 45000, Mumbai

I assumed that "ramesh Kumar" is a name that should be kept as is, with the first letter of the last name capitalized. The amount "45000" is a number that should be kept as is. The location "mumbai" might be a city name that should be capitalized, so I changed it to "Mumbai".

the problem is unpredictable and not machine-parseable!


In [30]:
# 1. Define messy invoice data
messy_invoices = [
    "Invoice #INV-2023-01 from Global Corp. on 2023-01-10. Total: $1250 for Services.",
    "Ref: 2023/B-002, dated 15.02.2023. Vendor: BIZ Solutions. Amt: 750 USD, Category: Software.",
    "Order #45678, dated Feb 28, 2023, for $300.00. Seller: Office Supplies Inc. Item: Stationery.",
    "Invoice-2023-004. Date: March 5th, 2023. Vendor: Data Analytics LLC. Price: 2500 USD for Consultancy.",
    "#INV-X-500 from Cloud Services Co. on 10/04/2023. Amount: 500.00 EUR for Hosting."
]

# 2. Define the strong system prompt for robust JSON extraction
#    This is crucial to ensure the LLM returns ONLY valid JSON.
strong_system_template = """You are a data extraction specialist for an accounting pipeline."
                  "Extract invoice data and return ONLY a valid JSON object. "
                  "DO NOT include any explanation, preamble, or markdown formatting. "
                  "Return only the JSON, nothing else."""

# 3. Define the JSON schema more accurately for the given data
invoice_schema_description = (
    """JSON schema (use null for missing values if not applicable, but try to infer):
    {
        \"invoice_id\": string,
        \"vendor_name\": string (Title Case),
        \"amount\": number (no currency symbols),
        \"currency\": string (e.g., USD, EUR, INR),
        \"invoice_date\": string (YYYY-MM-DD),
        \"category\": string (e.g., Services, Software, Stationery, Hosting, Consultancy, Other)
    }"""
)
invoice_system_prompt = strong_system_template + invoice_schema_description

# 4. Process each invoice string using the LLM
cleaned_invoice_data = []

print("--- Processing Invoices with LLM ---")
for i, invoice_text in enumerate(messy_invoices):
    print(f"\nProcessing invoice {i+1}/{len(messy_invoices)}...")
    user_prompt = f"Extract data from this invoice: {invoice_text}"

    # Make the LLM call with the strict system prompt and temperature=0.0
    llm_response = ask_llm(
        user_message=user_prompt,
        system_message=invoice_system_prompt,
        temperature=0.0
    )

    print(f"LLM Raw Response: {llm_response.strip()[:100]}...") # Print a snippet

    try:
        # Attempt to parse the LLM's response as JSON
        parsed_json = json.loads(llm_response.strip())
        cleaned_invoice_data.append(parsed_json)
        print("-> Successfully parsed JSON.")
    except json.JSONDecodeError as e:
        print(f"-> Failed to parse JSON: {e}. Attempting regex fallback...")
        # Fallback to regex if LLM adds extra text despite prompt
        match = re.search(r'\{.*?\}', llm_response, re.DOTALL)
        if match:
            try:
                parsed_json = json.loads(match.group())
                cleaned_invoice_data.append(parsed_json)
                print("-> Successfully parsed JSON with regex fallback.")
            except json.JSONDecodeError as e_fallback:
                print(f"--> Regex fallback also failed: {e_fallback}. Appending raw text for review.")
                cleaned_invoice_data.append({"raw_text": invoice_text, "llm_response_error": llm_response})
        else:
            print("--> No JSON object found with regex. Appending raw text for review.")
            cleaned_invoice_data.append({"raw_text": invoice_text, "llm_response_error": llm_response})

# 5. Convert the list of parsed JSON objects into a Pandas DataFrame
print("\n--- Creating DataFrame ---")
invoice_df = pd.DataFrame(cleaned_invoice_data)

# 6. Display the resulting clean DataFrame
print("\n--- Cleaned Invoice DataFrame ---")
print(invoice_df.to_string())

print("\nPipeline complete: Messy Text -> LLM -> JSON -> DataFrame -> Analysis (ready for next step).")

--- Processing Invoices with LLM ---

Processing invoice 1/5...
LLM Raw Response: {
    "invoice_id": "INV-2023-01",
    "vendor_name": "Global Corp.",
    "amount": 1250,
    "curre...
-> Successfully parsed JSON.

Processing invoice 2/5...
LLM Raw Response: {
    "invoice_id": "2023/B-002",
    "vendor_name": "Biz Solutions",
    "amount": 750,
    "curren...
-> Successfully parsed JSON.

Processing invoice 3/5...
LLM Raw Response: {
    "invoice_id": "45678",
    "vendor_name": "Office Supplies Inc.",
    "amount": 300.0,
    "cu...
-> Successfully parsed JSON.

Processing invoice 4/5...
LLM Raw Response: {
    "invoice_id": "Invoice-2023-004",
    "vendor_name": "Data Analytics LLC",
    "amount": 2500,...
-> Successfully parsed JSON.

Processing invoice 5/5...
LLM Raw Response: {
    "invoice_id": "#INV-X-500",
    "vendor_name": "Cloud Services Co.",
    "amount": 500.0,
    ...
-> Successfully parsed JSON.

--- Creating DataFrame ---

--- Cleaned Invoice DataFrame ---
         i